<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Telecom Site Power Prediction – Hybrid Engineering + ML Model
## Google Colab Python Code

# ============================================================
# TELECOM SITE POWER PREDICTION
# HYBRID ENGINEERING + ML MODEL
# C&W SEYCHELLES
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# ============================================================
# LOAD EXCEL FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"
site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"
traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ FILES
# ============================================================

site_db = pd.read_excel(site_db_url)
site_power = pd.read_excel(site_power_url)
traffic_4g = pd.read_excel(traffic_4g_url)
traffic_5g = pd.read_excel(traffic_5g_url)

In [ ]:
site_db.head(2)

In [ ]:
site_power.head(2)

In [ ]:
traffic_4g.head(2)

In [ ]:
traffic_5g.head(2)

In [ ]:
# ============================================================
# RENAME COLUMNS
# ============================================================

site_db.columns = [
    '#',
    'Site_ID',
    'Site_Name',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'AAU_5G',
    'Col_H',
    'Col_I',
    'Boards_4G',
    'Boards_5G',
    'BBU5900',
    'BBU3900',
    'BBU3910'
]

In [ ]:
# ============================================================
# AGGREGATE 4G TRAFFIC
# ============================================================

traffic_4g_summary = (

    traffic_4g.groupby(
        ["Site_ID", "trigger_ID", "date", "datetime"],
        as_index=False
    )["traffic_load_mbps"]

    .sum()

)

traffic_4g_summary.rename(
    columns={"traffic_load_mbps": "total_4g_traffic"},
    inplace=True
)
traffic_4g_summary.head(5)


In [ ]:
# ============================================================
# AGGREGATE 5G TRAFFIC
# ============================================================

traffic_5g_summary = (

    traffic_5g.groupby(
        ["Site_ID", "trigger_ID", "date", "datetime"],
        as_index=False
    )["traffic_load_mbps"]

    .sum()

)

traffic_5g_summary.rename(
    columns={"traffic_load_mbps": "total_5g_traffic"},
    inplace=True
)
traffic_5g_summary.head(5)


In [ ]:
# ============================================================
# MERGE DATASETS
# ============================================================

merged_df = site_power.merge(
    traffic_4g_summary,
    on=["Site_ID", "trigger_ID", "date", "datetime"],
    how="left"
)

merged_df = merged_df.merge(
    traffic_5g_summary,
    on=["Site_ID", "trigger_ID", "date", "datetime"],
    how="left"
)

merged_df = merged_df.merge(
    site_db,
    on="Site_ID",
    how="left"
)
merged_df.head(5)

In [ ]:
# ============================================================
# FILL NOMINAL ENGINEERING INDUSTRY VALUES
# ============================================================

merged_df.fillna(0, inplace=True)

# ============================================================
# ENGINEERING POWER ASSUMPTIONS
# ============================================================

RRU_2G_POWER = 150
RRU_3G_POWER = 200
RRU_4G_POWER = 180
AAU_5G_POWER = 500

BBU3900_POWER = 55
BBU3910_POWER = 65
BBU5900_POWER = 75

BOARD_4G_POWER = 42.5
BOARD_5G_POWER = 80
merged_df.head(5)

In [ ]:
# ============================================================
# STATIC POWER CALCULATIONS
# ============================================================

merged_df["power_2g"] = (
    merged_df["RRU_2G"] * RRU_2G_POWER
)

merged_df["power_3g"] = (
    merged_df["RRU_3G"] * RRU_3G_POWER
)

merged_df["power_4g_rru"] = (
    merged_df["RRU_4G"] * RRU_4G_POWER
)

merged_df["power_5g_aau"] = (
    merged_df["AAU_5G"] * AAU_5G_POWER
)

merged_df["power_bbu3900"] = (
    merged_df["BBU3900"] * BBU3900_POWER
)

merged_df["power_bbu3910"] = (
    merged_df["BBU3910"] * BBU3910_POWER
)

merged_df["power_bbu5900"] = (
    merged_df["BBU5900"] * BBU5900_POWER
)

merged_df["power_4g_boards"] = (
    merged_df["Boards_4G"] * BOARD_4G_POWER
)

merged_df["power_5g_boards"] = (
    merged_df["Boards_5G"] * BOARD_5G_POWER
)
merged_df.head(5)

In [ ]:
# ============================================================
# DYNAMIC TRAFFIC POWER
# ============================================================

merged_df["dynamic_4g_power"] = (
    merged_df["total_4g_traffic"] * 0.03
)

merged_df["dynamic_5g_power"] = (
    merged_df["total_5g_traffic"] * 0.04
)
merged_df.head(5)

In [ ]:
# ============================================================
# ENGINEERING PREDICTED POWER
# ============================================================

merged_df["engineering_predicted_power"] = (

    merged_df["power_2g"] +
    merged_df["power_3g"] +
    merged_df["power_4g_rru"] +
    merged_df["power_5g_aau"] +
    merged_df["power_bbu3900"] +
    merged_df["power_bbu3910"] +
    merged_df["power_bbu5900"] +
    merged_df["power_4g_boards"] +
    merged_df["power_5g_boards"] +
    merged_df["dynamic_4g_power"] +
    merged_df["dynamic_5g_power"]

)
merged_df.head(2)

In [ ]:
# ============================================================
# RESIDUAL ERROR
# ============================================================

merged_df["residual_error"] = (
    merged_df["site_power"] -
    merged_df["engineering_predicted_power"]
)
merged_df.head(2)

In [ ]:
# ============================================================
# MACHINE LEARNING FEATURES
# ============================================================

features = [

    "RRU_2G",
    "RRU_3G",
    "RRU_4G",
    "AAU_5G",
    "Boards_4G",
    "Boards_5G",
    "BBU3900",
    "BBU3910",
    "BBU5900",
    "total_4g_traffic",
    "total_5g_traffic",
    "engineering_predicted_power"

]
X = merged_df[features]

y = merged_df["residual_error"]

In [ ]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# ============================================================
# BASELINE MODEL
# ============================================================

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)

In [ ]:
# ============================================================
# IMPROVED MODEL
# ============================================================

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

In [ ]:
# ============================================================
# FINAL PREDICTIONS
# ============================================================

engineering_test = merged_df.loc[
    X_test.index,
    "engineering_predicted_power"
]

final_predictions = (
    engineering_test + rf_predictions
)

actual_values = merged_df.loc[
    X_test.index,
    "site_power"
]
merged_df.head(5)

In [ ]:
# ============================================================
# EVALUATION
# ============================================================

mae = mean_absolute_error(
    actual_values,
    final_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        actual_values,
        final_predictions
    )
)

mape = np.mean(
    np.abs(
        (actual_values - final_predictions)
        / actual_values
    )
) * 100

r2 = r2_score(
    actual_values,
    final_predictions
)

In [ ]:
# ============================================================
# PRINT RESULTS
# ============================================================

print("================================")
print("MODEL PERFORMANCE")
print("================================")

print(f"MAE  : {round(mae, 2)}")
print(f"RMSE : {round(rmse, 2)}")
print(f"MAPE : {round(mape, 2)} %")
print(f"R2   : {round(r2, 4)}")


In [ ]:
# ============================================================
# STORE FINAL RESULTS
# ============================================================

results_df = merged_df.loc[
    X_test.index,
    [
        "Site_ID",
        "trigger_ID",
        "date",
        "datetime",
        "site_power",
        "engineering_predicted_power"
    ]
].copy()

results_df["ml_correction"] = rf_predictions

results_df["final_predicted_power"] = final_predictions

results_df["error"] = (
    results_df["site_power"] -
    results_df["final_predicted_power"]
)

results_df["error_percentage"] = (

    np.abs(results_df["error"])
    /
    results_df["site_power"]

) * 100
results_df.head(5)

In [ ]:
# ============================================================
# EXPORT RESULTS
# ============================================================

results_df.to_excel(
    "Final_Site_Power_Predictions.xlsx",
    index=False
)

print("================================")
print("OUTPUT FILE CREATED")
print("================================")

print("Final_Site_Power_Predictions.xlsx")

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({

    "Feature": features,
    "Importance": rf_model.feature_importances_

})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("================================")
print("FEATURE IMPORTANCE")
print("================================")

print(importance_df)

In [ ]:
# ============================================================
# SAMPLE RESULTS
# ============================================================

print("================================")
print("SAMPLE PREDICTIONS")
print("================================")

print(results_df.head(20))

